# DhanHQ SDK - Complete Testing & Integration Guide

This notebook provides a comprehensive testing suite for all DhanHQ API functions.

**Table of Contents:**
1. Authentication & Setup
2. Fund Management
3. Portfolio Management
4. Order Management
5. Trade Book & History
6. Market Data (Real-time)
7. Historical Data
8. Option Chain Analysis
9. Security/Instrument List
10. Forever Orders (GTT)
11. eDIS & TPIN
12. Bulk Operations
13. Utility Functions

---
## 1. Authentication & Setup

Initialize the Dhan client and helper library.

In [7]:
# Import required libraries
import pandas as pd
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [8]:
# Initialize Dhan Client
from login import get_dhan_client
from lib.dhan_helper import DhanHelper

# Get authenticated client
dhan = get_dhan_client()
helper = DhanHelper(dhan)

print("✓ Dhan client initialized successfully")
print("✓ DhanHelper library loaded")
print("\n" + "="*80)
print("READY TO TEST ALL DHAN API FUNCTIONS")
print("="*80)
dhan.ohlc_data(
    securities = {"NSE_EQ":[1333]}
)

Using cached access token (Valid until: 2026-01-25T07:56:44)


2026-01-24 12:39:39,839 - INFO - Resolved Master List Path: c:\dhan_algo\master_list.csv
2026-01-24 12:39:39,842 - INFO - Loading master list from c:\dhan_algo\master_list.csv...
2026-01-24 12:39:39,846 - ERROR - Master list file not found: c:\dhan_algo\master_list.csv
2026-01-24 12:39:39,857 - WARNING - Master list not found or empty. Attempting to download...
2026-01-24 12:39:39,858 - INFO - Downloading compact master list for segments: ['NSE_EQ', 'NSE_FNO', 'BSE_EQ', 'BSE_FNO']...
2026-01-24 12:39:49,026 - INFO - Saved compact master list to c:\dhan_algo\master_list.csv (170609 records)
2026-01-24 12:39:49,069 - INFO - Master list downloaded successfully. Reloading...
2026-01-24 12:39:49,070 - INFO - Loading master list from c:\dhan_algo\master_list.csv...
2026-01-24 12:39:49,912 - INFO - Master list loaded: 170609 records
2026-01-24 12:40:48,101 - INFO - Master list indexed for faster lookup


✓ Dhan client initialized successfully
✓ DhanHelper library loaded

READY TO TEST ALL DHAN API FUNCTIONS


{'status': 'success',
 'remarks': '',
 'data': {'data': {'NSE_EQ': {'1333': {'last_price': 916.1,
     'ohlc': {'open': 920, 'close': 916.1, 'high': 926.1, 'low': 909.25}}}},
  'status': 'success'}}

In [11]:
dhan.quote_data(securities = {"NSE_EQ":[1333]})

{'status': 'success',
 'remarks': '',
 'data': {'data': {'NSE_EQ': {'1333': {'52_week_high': 1020.5,
     '52_week_low': 812.73,
     'average_price': 916.45,
     'buy_quantity': 0,
     'depth': {'buy': [{'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0}],
      'sell': [{'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0},
       {'quantity': 0, 'orders': 0, 'price': 0}]},
     'last_price': 916.1,
     'last_quantity': 1,
     'last_trade_time': '23/01/2026 15:59:51',
     'lower_circuit_limit': 826.85,
     'net_change': 0,
     'ohlc': {'open': 920, 'close': 916.1, 'high': 926.1, 'low': 909.25},
     'oi': 0,
     'oi_day_high': 0,
     'oi_day_low': 0,
     'sell_

---
## 2. Fund Management

Check available funds and margin details.

In [12]:
# Get available funds
available_funds = helper.get_available_funds()

print("FUND MANAGEMENT")
print("="*80)
print(f"Available Margin: Rs. {available_funds:,.2f}")
print("="*80)

FUND MANAGEMENT
Available Margin: Rs. 0.00


In [ ]:
# Get detailed fund limits (direct SDK call)
fund_limits = dhan.get_fund_limits()

if fund_limits.get('status') == 'success':
    data = fund_limits.get('data', {})
    print("\nDetailed Fund Limits:")
    print("-"*80)
    print(json.dumps(data, indent=2))
else:
    print(f"Error: {fund_limits.get('remarks')}")

---
## 3. Portfolio Management

View current positions and holdings.

In [ ]:
# Get current positions
positions = helper.get_positions()

print("CURRENT POSITIONS")
print("="*80)
print(f"Total Positions: {len(positions)}")

if not positions.empty:
    # Display key columns
    display_cols = ['tradingSymbol', 'positionType', 'netQty', 'buyAvg', 'sellAvg', 
                    'realizedProfit', 'unrealizedProfit']
    available_cols = [col for col in display_cols if col in positions.columns]
    print("\n", positions[available_cols])
else:
    print("No open positions")

In [ ]:
# Get holdings
holdings = helper.get_holdings()

print("\nHOLDINGS")
print("="*80)
print(f"Total Holdings: {len(holdings)}")

if not holdings.empty:
    # Display key columns
    display_cols = ['tradingSymbol', 'totalQty', 'avgCostPrice', 'currentPrice', 
                    'profitLoss', 'profitLossPercentage']
    available_cols = [col for col in display_cols if col in holdings.columns]
    print("\n", holdings[available_cols].head(10))
    
    # Summary statistics
    if 'profitLoss' in holdings.columns:
        total_pl = holdings['profitLoss'].sum()
        print(f"\nTotal P&L: Rs. {total_pl:,.2f}")
else:
    print("No holdings")

---
## 4. Order Management

Test order placement, modification, cancellation, and status tracking.

In [ ]:
# Get all orders for today
orders = helper.get_order_list()

print("ORDER BOOK")
print("="*80)
print(f"Total Orders Today: {len(orders)}")

if orders:
    # Convert to DataFrame for better display
    orders_df = pd.DataFrame(orders)
    display_cols = ['orderId', 'tradingSymbol', 'transactionType', 'orderStatus', 
                    'quantity', 'orderType', 'price']
    available_cols = [col for col in display_cols if col in orders_df.columns]
    print("\n", orders_df[available_cols].head(10))
else:
    print("No orders placed today")

In [ ]:
# Example: Place a test order (COMMENTED OUT - Uncomment to test)
# WARNING: This will place a real order!

# order_id = helper.place_order(
#     security_id='1333',  # HDFC Bank
#     exchange_segment=helper.NSE,
#     transaction_type=helper.BUY,
#     quantity=1,
#     order_type=helper.LIMIT,
#     product_type=helper.INTRA,
#     price=1500.00  # Set a price far from market to avoid execution
# )

# if order_id:
#     print(f"Order placed successfully! Order ID: {order_id}")
#     
#     # Get order status
#     status = helper.get_order_status(order_id)
#     print(f"Order Status: {status}")
#     
#     # Get full order details
#     order_details = helper.get_order_by_id(order_id)
#     print("\nOrder Details:")
#     print(json.dumps(order_details, indent=2))

print("Order placement example (commented out for safety)")
print("Uncomment the code above to test order placement")

In [ ]:
# Example: Modify order (COMMENTED OUT)
# order_id = "YOUR_ORDER_ID"
# success = helper.modify_order(
#     order_id=order_id,
#     quantity=2,
#     order_type=helper.LIMIT,
#     price=1450.00
# )
# print(f"Order modified: {success}")

print("Order modification example (commented out)")

In [ ]:
# Example: Cancel order (COMMENTED OUT)
# order_id = "YOUR_ORDER_ID"
# success = helper.cancel_order(order_id)
# print(f"Order cancelled: {success}")

print("Order cancellation example (commented out)")

---
## 5. Trade Book & History

View executed trades and historical trade data.

In [ ]:
# Get today's trade book
trades = helper.get_trade_book()

print("TRADE BOOK (TODAY)")
print("="*80)
print(f"Total Trades: {len(trades)}")

if trades:
    trades_df = pd.DataFrame(trades)
    display_cols = ['tradingSymbol', 'transactionType', 'quantity', 'tradedPrice', 
                    'tradedValue', 'exchangeTradeTime']
    available_cols = [col for col in display_cols if col in trades_df.columns]
    print("\n", trades_df[available_cols].head(10))
else:
    print("No trades executed today")

In [ ]:
# Get trade history for last 7 days
to_date = datetime.now().strftime("%Y-%m-%d")
from_date = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")

trade_history = helper.get_trade_history(from_date, to_date, page_number=0)

print(f"\nTRADE HISTORY ({from_date} to {to_date})")
print("="*80)
print(f"Total Trades: {len(trade_history)}")

if not trade_history.empty:
    display_cols = ['tradingSymbol', 'transactionType', 'quantity', 'tradedPrice', 
                    'tradedValue', 'tradeDate']
    available_cols = [col for col in display_cols if col in trade_history.columns]
    print("\n", trade_history[available_cols].head(10))
else:
    print("No trades in this period")

---
## 6. Market Data (Real-time)

Fetch live market data including LTP, OHLC, ticker, and quote data.

In [13]:
helper.find_index("NIFTY","NSE")["SECURITY_ID"]

13

In [14]:
# Get LTP using Explicit Arguments
print("LIVE MARKET DATA - LTP (EXPLICIT ARGUMENTS)")
print("="*80)

# Indices
nifty_ltp = helper.get_ltp("NIFTY", instrument="INDEX")
banknifty_ltp = helper.get_ltp("BANKNIFTY", instrument="INDEX")

print(f"Nifty 50: {nifty_ltp}")
print(f"Bank Nifty: {banknifty_ltp}")

# Stocks
hdfc_ltp = helper.get_ltp("HDFCBANK", instrument="EQUITY")
reliance_ltp = helper.get_ltp("RELIANCE", instrument="EQUITY")

print(f"\nHDFC Bank: {hdfc_ltp}")
print(f"Reliance: {reliance_ltp}")

# # Legacy ID check still supported (segment passed as exchange arg)
# legacy_nifty = helper.get_ltp("13", exchange="IDX_I")
# print(f"\nLegacy Check (Nifty ID 13): {legacy_nifty}")

LIVE MARKET DATA - LTP (EXPLICIT ARGUMENTS)
Nifty 50: 25048.65
Bank Nifty: 58473.1

HDFC Bank: 916.1
Reliance: 1386.1


In [ ]:
# Get OHLC data
print("\nOHLC DATA")
print("="*80)

hdfc_ohlc = helper.get_ohlc(1333, "NSE_EQ")
if hdfc_ohlc:
    print("HDFC Bank OHLC:")
    print(json.dumps(hdfc_ohlc, indent=2))
else:
    print("OHLC data not available (may require Data API subscription)")

In [ ]:
# Get ticker data for multiple securities
print("\nTICKER DATA (Multiple Securities)")
print("="*80)

ticker_data = helper.get_ticker_data({
    "NSE_EQ": [1333, 2885, 11915]  # HDFC, Reliance, TCS
})

if ticker_data:
    print(json.dumps(ticker_data, indent=2))
else:
    print("Ticker data not available (may require Data API subscription)")

In [ ]:
# Get quote data (OHLC + Volume + LTP)
print("\nQUOTE DATA")
print("="*80)

quote_data = helper.get_quote_data({
    "NSE_EQ": [1333]  # HDFC Bank
})

if quote_data:
    print(json.dumps(quote_data, indent=2))
else:
    print("Quote data not available (may require Data API subscription)")

---
## 7. Historical Data

Fetch historical daily and intraday data.

In [15]:
# Get daily historical data for Nifty (last 30 days)
to_date = datetime.now().strftime("%Y-%m-%d")
from_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

print(f"HISTORICAL DAILY DATA - Nifty 50 ({from_date} to {to_date})")
print("="*80)

daily_data = helper.get_historical_daily_data(
    security_id=13,
    exchange_segment="IDX_I",
    instrument_type="INDEX",
    from_date=from_date,
    to_date=to_date
)

if not daily_data.empty:
    print(f"Total Candles: {len(daily_data)}")
    print("\nLast 5 Days:")
    display_cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    available_cols = [col for col in display_cols if col in daily_data.columns]
    print(daily_data[available_cols].tail())
else:
    print("Historical data not available (requires Data API subscription)")

HISTORICAL DAILY DATA - Nifty 50 (2025-12-25 to 2026-01-24)
Total Candles: 20

Last 5 Days:
       timestamp      open      high       low     close       volume
15  1.768761e+09  25653.10  25653.30  25494.35  25585.50  443087953.0
16  1.768847e+09  25580.30  25585.00  25171.35  25232.50  409772324.0
17  1.768934e+09  25141.00  25300.95  24919.80  25157.50  395624981.0
18  1.769020e+09  25344.15  25435.75  25168.50  25289.90  486395433.0
19  1.769107e+09  25344.60  25347.95  25025.30  25048.65  393944908.0


In [16]:
# Get intraday 5-minute data for today
today = datetime.now().strftime("%Y-%m-%d")

print(f"\nINTRADAY 5-MINUTE DATA - Nifty 50 ({today})")
print("="*80)

intraday_data = helper.get_intraday_minute_data(
    security_id=13,
    exchange_segment="IDX_I",
    instrument_type="INDEX",
    interval="5",  # 1, 5, 15, 25, 60 minutes
    from_date=today,
    to_date=today
)

if not intraday_data.empty:
    print(f"Total Candles: {len(intraday_data)}")
    print("\nLast 5 Candles:")
    display_cols = ['timestamp', 'open', 'high', 'low', 'close', 'volume']
    available_cols = [col for col in display_cols if col in intraday_data.columns]
    print(intraday_data[available_cols].tail())
else:
    print("Intraday data not available (requires Data API subscription)")

2026-01-24 12:44:07,567 - ERROR - Failed to fetch intraday minute data: {'error_code': 'DH-905', 'error_type': 'Input_Exception', 'error_message': 'System is unable to fetch data due to incorrect parameters or no data present'}



INTRADAY 5-MINUTE DATA - Nifty 50 (2026-01-24)
Intraday data not available (requires Data API subscription)


In [ ]:
# Get expired options data
print("\nEXPIRED OPTIONS DATA")
print("="*80)

expired_data = helper.get_expired_options_data(
    security_id=13,  # Nifty
    exchange_segment="NSE_FNO",
    instrument_type="INDEX",
    expiry_flag="WEEK",  # WEEK or MONTH
    expiry_code=1,  # 1-5 for weekly, 1-3 for monthly
    strike="ATM",  # ATM, OTM1, OTM2, ITM1, ITM2, or specific price
    drv_option_type="CALL",  # CALL or PUT
    required_data=["open", "high", "low", "close", "volume", "oi"],
    from_date=(datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d"),
    to_date=datetime.now().strftime("%Y-%m-%d")
)

if not expired_data.empty:
    print(f"Total Records: {len(expired_data)}")
    print("\nSample Data:")
    print(expired_data.head())
else:
    print("Expired options data not available (requires Data API subscription)")

---
## 8. Option Chain Analysis

Fetch expiry dates and complete option chain data.

In [ ]:
# Get expiry list for Nifty
print("OPTION CHAIN - Nifty 50")
print("="*80)

expiry_list = helper.get_expiry_list(
    under_security_id=13,  # Nifty
    under_exchange_segment="IDX_I"
)

print(f"Available Expiries: {len(expiry_list)}")
print("\nExpiry Dates:")
for i, expiry in enumerate(expiry_list[:5], 1):
    print(f"{i}. {expiry}")

if expiry_list:
    nearest_expiry = expiry_list[0]
    print(f"\nNearest Expiry: {nearest_expiry}")
else:
    print("\nNo expiries available")

In [ ]:
# Get option chain for nearest expiry
if expiry_list:
    nearest_expiry = expiry_list[0]
    
    print(f"\nOPTION CHAIN for {nearest_expiry}")
    print("="*80)
    
    option_chain = helper.get_option_chain(
        under_security_id=13,
        expiry=nearest_expiry,
        under_exchange_segment="IDX_I"
    )
    
    if not option_chain.empty:
        print(f"Total Strikes: {len(option_chain)}")
        print("\nSample Option Chain Data:")
        print(option_chain.head(10))
        
        # Show column names
        print("\nAvailable Columns:")
        print(option_chain.columns.tolist())
    else:
        print("Option chain data not available")
else:
    print("No expiries available to fetch option chain")

In [ ]:
# Get Bank Nifty expiries and option chain
print("\nOPTION CHAIN - Bank Nifty")
print("="*80)

banknifty_expiries = helper.get_expiry_list(
    under_security_id=25,  # Bank Nifty
    under_exchange_segment="IDX_I"
)

print(f"Available Expiries: {len(banknifty_expiries)}")
if banknifty_expiries:
    print(f"Nearest Expiry: {banknifty_expiries[0]}")

---
## 9. Security/Instrument List

Fetch the complete security master list.

In [ ]:
# Get compact security list
print("SECURITY MASTER LIST (Compact)")
print("="*80)

securities = helper.fetch_security_list("compact")

if not securities.empty:
    print(f"Total Securities: {len(securities)}")
    print("\nSample Securities:")
    display_cols = ['SEM_SMST_SECURITY_ID', 'SEM_TRADING_SYMBOL', 
                    'SEM_EXCH_INSTRUMENT_TYPE', 'SEM_EXPIRY_DATE']
    available_cols = [col for col in display_cols if col in securities.columns]
    print(securities[available_cols].head(10))
    
    print("\nAvailable Columns:")
    print(securities.columns.tolist())
else:
    print("Security list not available")

In [ ]:
# Search for specific securities
if not securities.empty and 'SEM_TRADING_SYMBOL' in securities.columns:
    print("\nSEARCH SECURITIES")
    print("="*80)
    
    # Search for HDFC
    hdfc_securities = securities[securities['SEM_TRADING_SYMBOL'].str.contains('HDFC', na=False)]
    print(f"\nSecurities containing 'HDFC': {len(hdfc_securities)}")
    print(hdfc_securities[available_cols].head())

---
## 9.5 Optimized Instrument Lookups

These functions use a pre-indexed master list and optimized filtering order (Exchange -> Instrument -> Underlying -> Expiry) for high-performance scanning.

In [ ]:
print("OPTIMIZED LOOKUPS")
print("="*80)

# 1. Fast Equity Lookup
tcs_sec = helper.find_equity("TCS")
if tcs_sec:
    print(f"Equity: {tcs_sec['SYMBOL_NAME']} | ID: {tcs_sec['SECURITY_ID']}")

# 2. Fast Future Lookup (Nearest Expiry)
nifty_fut = helper.find_future("NIFTY")
if nifty_fut:
    print(f"Future: {nifty_fut['SYMBOL_NAME']} | Expiry: {nifty_fut['SM_EXPIRY_DATE']}")
    
# 3. Fast Option Lookup
if nifty_fut:
    expiry = nifty_fut['SM_EXPIRY_DATE']
    # Get current Nifty LTP for strike selection (using the variable from previous cells)
    try:
        # Use 50-point rounding for Nifty
        ref_price = nifty_ltp if 'nifty_ltp' in locals() and nifty_ltp > 0 else 24000
        strike = round(ref_price / 50) * 50
        
        nifty_opt = helper.find_option("NIFTY", expiry, strike, "CE")
        if nifty_opt:
            print(f"Option: {nifty_opt['SYMBOL_NAME']} | Strike: {nifty_opt['STRIKE_PRICE']}")
    except Exception as e:
        print(f"Could not select ATM strike: {e}")
        # Fallback to a hardcoded strike if necessary
        nifty_opt = helper.find_option("NIFTY", expiry, 24000, "CE")
        if nifty_opt:
            print(f"Option (Fallback): {nifty_opt['SYMBOL_NAME']}")

---
## 10. Forever Orders (GTT)

Place Good-Till-Triggered orders.

In [ ]:
# Example: Place Forever Order (COMMENTED OUT)
# WARNING: This will place a real GTT order!

# gtt_order_id = helper.place_forever_order(
#     security_id="1333",  # HDFC Bank
#     exchange_segment=helper.NSE,
#     transaction_type=helper.BUY,
#     quantity=10,
#     price=1900,
#     trigger_price=1950,
#     order_type=helper.LIMIT,
#     product_type=helper.CNC
# )

# if gtt_order_id:
#     print(f"Forever Order placed! Order ID: {gtt_order_id}")

print("FOREVER ORDERS (GTT)")
print("="*80)
print("Forever order placement example (commented out for safety)")
print("Uncomment the code above to test GTT order placement")

---
## 11. eDIS & TPIN

Manage eDIS authorization for selling holdings.

In [ ]:
# Generate TPIN
print("eDIS & TPIN MANAGEMENT")
print("="*80)

# Example: Generate TPIN (COMMENTED OUT)
# success = helper.generate_tpin()
# print(f"TPIN generation triggered: {success}")

print("TPIN generation example (commented out)")
print("Uncomment to trigger TPIN generation")

In [ ]:
# Check eDIS status for a specific ISIN
# Example ISIN: 'INE00IN01015' (adjust as needed)

# edis_status = helper.get_edis_status(isin='INE00IN01015')
# if edis_status:
#     print("eDIS Status:")
#     print(json.dumps(edis_status, indent=2))

print("\neDIS status check example (commented out)")
print("Provide a valid ISIN to check status")

---
## 12. Bulk Operations

Perform bulk actions on orders and positions.

In [ ]:
# Example: Cancel all pending orders (COMMENTED OUT)
# WARNING: This will cancel ALL pending orders!

# cancelled_count = helper.cancel_all_orders()
# print(f"Cancelled {cancelled_count} orders")

print("BULK OPERATIONS")
print("="*80)
print("Cancel all orders example (commented out for safety)")
print("Uncomment to cancel all pending orders")

In [ ]:
# Example: Close all positions (COMMENTED OUT)
# WARNING: This will close ALL open positions!

# closed_count = helper.close_all_positions()
# print(f"Closed {closed_count} positions")

print("\nClose all positions example (commented out for safety)")
print("Uncomment to close all open positions")

---
## 13. Utility Functions

Helper utilities for time conversion and constants.

In [ ]:
# Epoch to DateTime conversion
print("UTILITY FUNCTIONS")
print("="*80)

epoch_time = 1706000000
human_time = helper.epoch_to_datetime(epoch_time)
print(f"Epoch {epoch_time} -> {human_time}")

In [ ]:
# Available constants
print("\nAVAILABLE CONSTANTS")
print("="*80)

print("Exchange Segments:")
print(f"  NSE: {helper.NSE}")
print(f"  BSE: {helper.BSE}")
print(f"  NSE_FNO: {helper.NSE_FNO}")

print("\nTransaction Types:")
print(f"  BUY: {helper.BUY}")
print(f"  SELL: {helper.SELL}")

print("\nOrder Types:")
print(f"  MARKET: {helper.MARKET}")
print(f"  LIMIT: {helper.LIMIT}")

print("\nProduct Types:")
print(f"  INTRA: {helper.INTRA}")
print(f"  CNC: {helper.CNC}")
print(f"  MARGIN: {helper.MARGIN}")

---
## Summary

This notebook demonstrates all available DhanHQ SDK functions through the DhanHelper library.

### Key Takeaways:

1. **Authentication**: Handled automatically via `login.py`
2. **Fund Management**: Check available margin and limits
3. **Portfolio**: View positions and holdings
4. **Orders**: Place, modify, cancel, and track orders
5. **Trades**: Access trade book and history
6. **Market Data**: Real-time LTP, OHLC, ticker, and quote data
7. **Historical Data**: Daily and intraday candles, expired options
8. **Option Chain**: Complete option chain with Greeks and OI
9. **Security List**: Master list of all tradable instruments
10. **Forever Orders**: GTT orders for swing trading
11. **eDIS**: Manage holdings authorization
12. **Bulk Operations**: Cancel all orders or close all positions
13. **Utilities**: Time conversion and constants

### Important Notes:

- Some functions require **Data API subscription** (historical data, market quotes)
- Order placement examples are **commented out** for safety
- All functions include **error handling** and return safe defaults
- Use **type hints** for better IDE support

### Next Steps:

1. Uncomment and test order placement in a paper trading environment
2. Subscribe to Data APIs for full market data access
3. Build custom strategies using these helper functions
4. Integrate with your existing algo trading framework

---

**Documentation**: See `DHAN_HELPER_REFERENCE.md` for complete API reference

**Quick Reference**: See `dhan_helper_quick_ref.py` for code snippets